# Load Libraries

In [17]:
import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter
from nltk.util import ngrams

In [18]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# Load Data

In [19]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# 1. Filtering Data

In [20]:
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [21]:
print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nClass Distribution:")
print(df['label'].value_counts())

Dataset Shape: (5572, 2)

Missing Values:
label      0
message    0
dtype: int64

Class Distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


## 2. Text Preprocessing

Text preprocessing is an important step in NLP because raw text may contain unnecessary characters and words that can affect the analysis.

In this project, I applied the following preprocessing steps:

1. **Lowercasing:** Convert all text into lowercase to make words consistent.
2. **Removing Numbers:** Remove numerical values because they are not needed for our text analysis.
3. **Removing Punctuation:** Remove punctuation marks such as commas, periods, and question marks.
4. **Removing Special Characters:** Remove special characters such as currency symbols and other non-alphabetic characters.
5. **Tokenization:** Split each message into individual words using NLTK's `word_tokenize()`.
6. **Stopwords Removal:** Remove common words that do not add significant meaning to the analysis.
7. **Lemmatization:** Convert words into their base form using WordNetLemmatizer.

After preprocessing, each message is represented as a list of clean tokens that can be used in the N-Gram Language Model.

In [22]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

   # Remove numbers, punctuation and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

In [23]:
df['processed_message'] = df['message'].apply(preprocess_text)
df.head()

,label,message,processed_message
0,ham,"Go until jurong point, crazy.. Available only ...","[go, jurong, point, crazy, available, bugis, n..."
1,ham,Ok lar... Joking wif u oni...,"[ok, lar, joking, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, wkly, comp, win, fa, cup, final,..."
3,ham,U dun say so early hor... U c already then say...,"[u, dun, say, early, hor, u, c, already, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...","[nah, think, go, usf, life, around, though]"


In [24]:
print("Original Message:")
print(df['message'].iloc[0])

print("\nProcessed Message:")
print(df['processed_message'].iloc[0])

Original Message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Processed Message:
['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']


## 3. N-Gram Language Model

An N-Gram Language Model is a statistical language model that predicts the probability of a word based on the previous words.

In this project, I used a **Bigram Model (N=2)**, which considers two consecutive words at a time.

In [25]:
all_tokens = []

for tokens in df['processed_message']:
    all_tokens.extend(tokens)

print("Total number of tokens:", len(all_tokens))
print("First 20 tokens:")
print(all_tokens[:20])

Total number of tokens: 50035
First 20 tokens:
['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat', 'ok', 'lar', 'joking', 'wif']


### 3.1 Unigram Model

A Unigram represents a single word.

The frequency of each word is calculated using the Counter function.

In [26]:
unigram_counts = Counter(all_tokens)
print("Most common 10 words:")
print(unigram_counts.most_common(10))

Most common 10 words:
[('u', 1279), ('call', 639), ('get', 408), ('ur', 385), ('gt', 318), ('lt', 316), ('go', 312), ('ok', 292), ('day', 291), ('free', 288)]


### 3.2 Bigram Model

A Bigram represents a pair of consecutive words.

For example, the sentence:

`I love machine learning`

Produces the following bigrams:

- (I, love)
- (love, machine)
- (machine, learning)

In [27]:
all_bigrams = []

for tokens in df['processed_message']:
    message_bigrams = list(ngrams(tokens, 2))
    all_bigrams.extend(message_bigrams)

bigram_counts = Counter(all_bigrams)

print("Most common 10 bigrams:")
print(bigram_counts.most_common(10))

Most common 10 bigrams:
[(('lt', 'gt'), 276), (('gon', 'na'), 58), (('please', 'call'), 55), (('call', 'later'), 52), (('co', 'uk'), 49), (('u', 'r'), 46), (('let', 'know'), 41), (('r', 'u'), 39), (('take', 'care'), 39), (('sorry', 'call'), 38)]


### 3.3 Conditional Probability

Conditional probability measures how likely a word is to appear after another word.

Since we are using a Bigram Model, each word depends only on the previous word.

The probability is calculated using:

$$
P(w_2 \mid w_1)
=
\frac{Count(w_1, w_2)}{Count(w_1)}
$$

Where:

- `Count(w1, w2)`: Number of occurrences of the bigram.
- `Count(w1)`: Number of times the first word appears as a context.

This probability is used to calculate the probability of a complete sentence.

### Calculating Context Frequencies

To calculate conditional probabilities, we need to know how many times each word appears as the first word in a bigram.

Therefore, we count all words except the last word in each message.

These counts represent the possible contexts used in the Bigram Model.

In [28]:
context_counts = Counter()

for tokens in df['processed_message']:
    context_counts.update(tokens[:-1])

## 4. Sentence Probability Calculation

In this step, we select three messages from the dataset to calculate their probabilities using the Bigram Language Model.

Only messages containing at least two processed words are selected because the Bigram Model requires at least two words to calculate conditional probabilities.

We use the original messages for display and their processed tokens for probability calculations.

In [ ]:
selected_messages = df[df['processed_message'].apply(lambda x: len(x) >= 2)].head(3)

for index, row in selected_messages.iterrows():
    print("Original Message:", row['message'])
    print("Processed Message:", row['processed_message'])
    print("-" * 50)

Original Message: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Processed Message: ['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']
--------------------------------------------------
Original Message: Ok lar... Joking wif u oni...
Processed Message: ['ok', 'lar', 'joking', 'wif', 'u', 'oni']
--------------------------------------------------
Original Message: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Processed Message: ['free', 'entry', 'wkly', 'comp', 'win', 'fa', 'cup', 'final', 'tkts', 'st', 'may', 'text', 'fa', 'receive', 'entry', 'question', 'std', 'txt', 'rate', 'c', 'apply']
--------------------------------------------------


### 4.1 Calculating Sentence Probability

The probability of a sentence is calculated using the Markov Assumption.

First, we calculate the probability of the first word using the Unigram Model.

Then, we calculate the conditional probability of each following word based on the previous word using the Bigram Model.

The final sentence probability is calculated by multiplying all these probabilities together.

The formula is:

$$
P(w_1, w_2, \ldots, w_n)
=
P(w_1) \times
\prod_{i=2}^{n} P(w_i \mid w_{i-1})
$$
This allows us to estimate how likely a sentence is to occur based on the word frequencies and relationships learned from the dataset.

In [30]:
def calculate_sentence_probability(tokens):

    total_tokens = len(all_tokens)

    # Calculate probability of the first word
    first_word = tokens[0]
    first_word_probability = unigram_counts[first_word] / total_tokens

    sentence_probability = first_word_probability

    print(f"P({first_word}) = {unigram_counts[first_word]} / {total_tokens} = {first_word_probability:.8f}")

    # Calculate probability of each next word
    for i in range(1, len(tokens)):

        previous_word = tokens[i - 1]
        current_word = tokens[i]

        bigram_count = bigram_counts[(previous_word, current_word)]
        context_count = context_counts[previous_word]

        if context_count == 0:
            word_probability = 0
        else:
            word_probability = bigram_count / context_count

        print(
            f"P({current_word} | {previous_word}) = "
            f"{bigram_count} / {context_count} = {word_probability:.8f}"
        )

        sentence_probability *= word_probability

    return sentence_probability

### 4.2 Probability Results

Finally, we calculate the probability of each selected message.

For every message, we display:

- The original sentence.
- The processed sentence.
- The probability of the first word.
- The conditional probability of each following word.
- The final sentence probability.

In [31]:
for index, row in selected_messages.iterrows():

    tokens = row['processed_message']

    print("\nSentence:", row['message'])
    print("Processed Sentence:", tokens)
    print("\nCalculation Steps:")

    probability = calculate_sentence_probability(tokens)

    print("\nFinal Sentence Probability:", f"{probability:.10e}")
    print("=" * 70)


Sentence: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Processed Sentence: ['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']

Calculation Steps:
P(go) = 312 / 50035 = 0.00623564
P(jurong | go) = 1 / 304 = 0.00328947
P(point | jurong) = 1 / 1 = 1.00000000
P(crazy | point) = 1 / 29 = 0.03448276
P(available | crazy) = 1 / 14 = 0.07142857
P(bugis | available) = 1 / 16 = 0.06250000
P(n | bugis) = 1 / 5 = 0.20000000
P(great | n) = 1 / 170 = 0.00588235
P(world | great) = 1 / 111 = 0.00900901
P(la | world) = 1 / 38 = 0.02631579
P(e | la) = 1 / 8 = 0.12500000
P(buffet | e) = 1 / 96 = 0.01041667
P(cine | buffet) = 1 / 2 = 0.50000000
P(got | cine) = 1 / 7 = 0.14285714
P(amore | got) = 1 / 244 = 0.00409836
P(wat | amore) = 1 / 1 = 1.00000000

Final Sentence Probability: 3.3570399295e-22

Sentence: Ok lar... Joking wif u oni...
Processed Sentence: ['o